# CancerNet: Breast Cancer Histology Classification with a CNN
### IDC (Invasive Ductal Carcinoma) Detection — Benign vs. Malignant

**Objective:** Build a breast cancer classifier on the IDC dataset that can accurately classify a histology image patch as benign or malignant.

**Business Goal:** Develop a Convolutional Neural Network (CNN), referred to as **CancerNet**, using Keras/TensorFlow, trained on 50×50 histology image patches, to achieve high classification accuracy.

**Dataset:** IDC_regular breast cancer histology dataset (Kaggle) — 277,524 patches of size 50×50, extracted from 162 whole-mount slide images of breast cancer specimens scanned at 40x. 198,738 patches are IDC-negative and 78,786 are IDC-positive.
Download: https://www.kaggle.com/datasets/paultimothymooney/breast-histopathology-images (requires ~3.02 GB disk space).

**Environment:** This notebook is designed to run in **Google Colab (GPU runtime)** or a local Jupyter environment with TensorFlow/Keras and a GPU. Training on the full 277K-image dataset on CPU only is impractically slow — a GPU is strongly recommended.


## 0. Setup — Install & Import Dependencies

In [ ]:
# Run this cell first. If in Colab, also go to Runtime > Change runtime type > GPU.
!pip install tensorflow scikit-learn matplotlib seaborn opencv-python-headless --quiet

import os, glob, random, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, BatchNormalization,
                                      Activation, Dropout, Flatten, Dense,
                                      SeparableConv2D, Input)
from tensorflow.keras.optimizers import Adagrad, Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, roc_auc_score, roc_curve)
from sklearn.utils import class_weight

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)


: 

## 1. Data Preparation

After downloading and extracting the IDC_regular dataset, the folder structure looks like:

```
IDC_regular_ps50_idx5/
    <patient_id>/
        0/   <- benign (non-IDC) 50x50 patches
        1/   <- malignant (IDC) 50x50 patches
```

Update `DATA_DIR` below to point at wherever you extracted the dataset.


In [ ]:
DATA_DIR = "/content/IDC_regular_ps50_idx5"   # <-- UPDATE THIS PATH

# Recursively collect every image path and its label (0 = benign, 1 = malignant)
image_paths = glob.glob(os.path.join(DATA_DIR, "**", "*.png"), recursive=True)
print(f"Found {len(image_paths):,} total image patches")

data = []
for path in image_paths:
    label = int(os.path.basename(path).split("class")[-1][0]) if "class" in path else int(os.path.basename(os.path.dirname(path)))
    data.append({"path": path, "label": label})

df = pd.DataFrame(data)
print(df["label"].value_counts())
print(f"\nClass balance: {df['label'].value_counts(normalize=True).round(3).to_dict()}")


In [ ]:
# Visualize a few sample patches from each class
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for cls in [0, 1]:
    samples = df[df.label == cls].sample(6, random_state=SEED)
    for i, (_, row) in enumerate(samples.iterrows()):
        img = cv2.cvtColor(cv2.imread(row["path"]), cv2.COLOR_BGR2RGB)
        axes[cls, i].imshow(img)
        axes[cls, i].axis("off")
    axes[cls, 0].set_ylabel("Benign" if cls == 0 else "Malignant", fontsize=12)
fig.suptitle("Sample Histology Patches: Benign (top) vs. Malignant (bottom)")
plt.tight_layout()
plt.savefig("sample_patches.png", dpi=150)
plt.show()


### Train / Validation / Test Split

**Answer to Q1 (train/test split used):** We use a **70% train / 10% validation / 20% test** split, stratified by class to preserve the ~72:28 benign:malignant ratio in every split. The validation set is used for early stopping and learning-rate scheduling during training; the test set is held out completely until final evaluation.


In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.20, stratify=df["label"], random_state=SEED
)
train_df, val_df = train_test_split(
    train_df, test_size=0.125, stratify=train_df["label"], random_state=SEED
)  # 0.125 x 0.80 = 0.10 of the full dataset -> final split is 70/10/20

print(f"Train: {len(train_df):,}  ({len(train_df)/len(df):.1%})")
print(f"Val:   {len(val_df):,}  ({len(val_df)/len(df):.1%})")
print(f"Test:  {len(test_df):,}  ({len(test_df)/len(df):.1%})")

for name, d in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"{name} class balance:", d["label"].value_counts(normalize=True).round(3).to_dict())


In [ ]:
# Class weights to handle the ~72:28 imbalance (helps recall on the minority malignant class)
class_weights = class_weight.compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)
class_weight_dict = dict(enumerate(class_weights))
print("Class weights:", class_weight_dict)


In [ ]:
IMG_SIZE = 50
BATCH_SIZE = 32

train_df["label_str"] = train_df["label"].astype(str)
val_df["label_str"] = val_df["label"].astype(str)
test_df["label_str"] = test_df["label"].astype(str)

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    vertical_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_dataframe(
    train_df, x_col="path", y_col="label_str",
    target_size=(IMG_SIZE, IMG_SIZE), class_mode="binary",
    batch_size=BATCH_SIZE, shuffle=True, seed=SEED
)
val_gen = val_test_datagen.flow_from_dataframe(
    val_df, x_col="path", y_col="label_str",
    target_size=(IMG_SIZE, IMG_SIZE), class_mode="binary",
    batch_size=BATCH_SIZE, shuffle=False
)
test_gen = val_test_datagen.flow_from_dataframe(
    test_df, x_col="path", y_col="label_str",
    target_size=(IMG_SIZE, IMG_SIZE), class_mode="binary",
    batch_size=BATCH_SIZE, shuffle=False
)


## 2. Model Architecture — CancerNet

CancerNet is a compact VGG-style CNN built with **separable convolutions** (`SeparableConv2D`), which use far fewer parameters than standard convolutions while retaining strong performance — well suited to a large dataset of small (50×50) images. The architecture follows a widening pattern (32 → 64 → 128 filters) with batch normalization, max pooling, and dropout after each block to control overfitting, followed by two dense layers and a sigmoid output for binary classification.


In [ ]:
def build_cancernet(width=50, height=50, depth=3, classes=1):
    model = Sequential(name="CancerNet")
    input_shape = (height, width, depth)
    chan_dim = -1  # channels-last

    model.add(Input(shape=input_shape))

    # ---- Block 1: 32 filters ----
    model.add(SeparableConv2D(32, (3, 3), padding="same"))
    model.add(Activation("relu"))
    model.add(BatchNormalization(axis=chan_dim))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    # ---- Block 2: 64 filters (x2 conv) ----
    model.add(SeparableConv2D(64, (3, 3), padding="same"))
    model.add(Activation("relu"))
    model.add(BatchNormalization(axis=chan_dim))
    model.add(SeparableConv2D(64, (3, 3), padding="same"))
    model.add(Activation("relu"))
    model.add(BatchNormalization(axis=chan_dim))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    # ---- Block 3: 128 filters (x3 conv) ----
    model.add(SeparableConv2D(128, (3, 3), padding="same"))
    model.add(Activation("relu"))
    model.add(BatchNormalization(axis=chan_dim))
    model.add(SeparableConv2D(128, (3, 3), padding="same"))
    model.add(Activation("relu"))
    model.add(BatchNormalization(axis=chan_dim))
    model.add(SeparableConv2D(128, (3, 3), padding="same"))
    model.add(Activation("relu"))
    model.add(BatchNormalization(axis=chan_dim))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    # ---- Classifier head ----
    model.add(Flatten())
    model.add(Dense(256))
    model.add(Activation("relu"))
    model.add(BatchNormalization())
    model.add(Dropout(0.5))

    model.add(Dense(classes))
    model.add(Activation("sigmoid"))

    return model

model = build_cancernet(width=IMG_SIZE, height=IMG_SIZE, depth=3, classes=1)
model.summary()


In [ ]:
model.compile(
    loss="binary_crossentropy",
    optimizer=Adagrad(learning_rate=1e-2),
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall"),
             tf.keras.metrics.AUC(name="auc")]
)


## 3. Training

**Answer to Q2 (epochs/iterations):** We train for up to **40 epochs**, but use `EarlyStopping` (patience=5, monitored on validation loss) so training stops automatically once the model stops improving — in practice this dataset typically converges and early-stops well before 40 epochs. `ReduceLROnPlateau` reduces the learning rate when validation loss plateaus, and `ModelCheckpoint` saves only the best-performing weights.

We also explicitly record **accuracy at epoch 5 and epoch 10** below (Q4) by inspecting the training history after the run.


In [ ]:
EPOCHS = 40

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint("cancernet_best.keras", monitor="val_loss", save_best_only=True, verbose=1),
]

history = model.fit(
    train_gen,
    steps_per_epoch=len(train_gen),
    validation_data=val_gen,
    validation_steps=len(val_gen),
    epochs=EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)


In [ ]:
# ---- Q4: Accuracy after 5 epochs and 10 epochs ----
hist = history.history
print("=== Accuracy by epoch checkpoint ===")
if len(hist["accuracy"]) >= 5:
    print(f"After  5 epochs -> train_acc: {hist['accuracy'][4]:.4f}   val_acc: {hist['val_accuracy'][4]:.4f}")
if len(hist["accuracy"]) >= 10:
    print(f"After 10 epochs -> train_acc: {hist['accuracy'][9]:.4f}   val_acc: {hist['val_accuracy'][9]:.4f}")
print(f"\nTraining stopped after {len(hist['accuracy'])} epochs "
      f"(EarlyStopping patience=5 on val_loss).")


In [ ]:
# ---- Training curves ----
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(hist["accuracy"], label="Train Accuracy")
axes[0].plot(hist["val_accuracy"], label="Validation Accuracy")
axes[0].axvline(4, color="gray", linestyle="--", alpha=0.5, label="Epoch 5")
axes[0].axvline(9, color="gray", linestyle=":", alpha=0.5, label="Epoch 10")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy")
axes[0].set_title("CancerNet Accuracy per Epoch")
axes[0].legend()

axes[1].plot(hist["loss"], label="Train Loss")
axes[1].plot(hist["val_loss"], label="Validation Loss")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Binary Cross-Entropy Loss")
axes[1].set_title("CancerNet Loss per Epoch")
axes[1].legend()

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()


## 4. Model Evaluation

Evaluate the trained model on the held-out **test set** (never seen during training or validation) using accuracy, precision, recall, F1-score, and ROC-AUC.


In [ ]:
test_gen.reset()
y_pred_proba = model.predict(test_gen, steps=len(test_gen), verbose=1).flatten()
y_pred = (y_pred_proba > 0.5).astype(int)
y_true = test_gen.classes  # ground-truth labels in the order Keras generated predictions

test_accuracy = accuracy_score(y_true, y_pred)
test_auc = roc_auc_score(y_true, y_pred_proba)

print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test ROC-AUC:  {test_auc:.4f}\n")
print(classification_report(y_true, y_pred, target_names=["Benign (0)", "Malignant (1)"]))


In [ ]:
# ROC curve
fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
plt.figure(figsize=(5.5, 5))
plt.plot(fpr, tpr, label=f"CancerNet (AUC = {test_auc:.3f})", linewidth=2)
plt.plot([0, 1], [0, 1], "k--", label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — CancerNet on IDC Test Set")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("roc_curve.png", dpi=150)
plt.show()


## 5. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

plt.figure(figsize=(5.5, 4.8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Benign", "Malignant"], yticklabels=["Benign", "Malignant"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — CancerNet Test Set Predictions")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

print(f"True Negatives  (benign correctly identified):    {tn}")
print(f"False Positives (benign misclassified as IDC):     {fp}")
print(f"False Negatives (IDC missed / misclassified):       {fn}  <-- most clinically costly error")
print(f"True Positives  (IDC correctly identified):         {tp}")

sensitivity = tp / (tp + fn)   # a.k.a. recall — critical metric for cancer screening
specificity = tn / (tn + fp)
print(f"\nSensitivity (Recall on malignant): {sensitivity:.4f}")
print(f"Specificity (Recall on benign):     {specificity:.4f}")


## 6. Performance Analysis

**Interpreting the confusion matrix for this clinical context:**

- **False Negatives (FN)** — malignant patches the model misses — are the most dangerous error type, since they represent a missed cancer diagnosis. This is why **Recall/Sensitivity on the malignant class** is emphasized as a key metric, often even above raw accuracy.
- **False Positives (FP)** — benign patches flagged as malignant — are less dangerous clinically (they typically trigger a follow-up review by a pathologist) but too many of them reduce the model's practical usefulness by creating alert fatigue.
- Given the ~72:28 class imbalance in the raw dataset, plain accuracy can be misleading (a model that always predicts "benign" would already score ~72%). This is why class weighting was applied during training and why precision/recall/F1/AUC are reported alongside accuracy.

Fill in this cell's discussion with your actual numbers after running the notebook — e.g., "The model achieved a sensitivity of X%, meaning Y% of malignant patches were correctly flagged, with Z false negatives out of N malignant test patches."


In [ ]:
# Save the final model and a summary of results for reporting
model.save("cancernet_final.keras")

summary = {
    "test_accuracy": float(test_accuracy),
    "test_auc": float(test_auc),
    "sensitivity_recall_malignant": float(sensitivity),
    "specificity_recall_benign": float(specificity),
    "confusion_matrix": cm.tolist(),
    "epochs_trained": len(hist["accuracy"]),
    "acc_at_epoch_5": float(hist["accuracy"][4]) if len(hist["accuracy"]) >= 5 else None,
    "acc_at_epoch_10": float(hist["accuracy"][9]) if len(hist["accuracy"]) >= 10 else None,
}
import json
with open("cancernet_results_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))


## 7. Project Questions — Answers

**1. What is the training and testing split you used?**
70% train / 10% validation / 20% test, stratified by class label to preserve the benign:malignant ratio in each split.

**2. How many epochs / iterations did you run your model?**
Up to 40 epochs, with `EarlyStopping` (patience 5 on validation loss) so training halts automatically once the model stops improving — see the printed "Training stopped after N epochs" output above for the actual number reached on your run.

**3. Do you think CNN is best for image datasets, or are there better algorithms?**
CNNs remain the strongest default choice for image classification because their convolutional filters exploit spatial locality and translation invariance — properties classical ML algorithms (Logistic Regression, SVM, Random Forest) don't have unless features are hand-engineered first. For histology images specifically, a few alternatives/extensions are worth naming:
- **Transfer learning with pretrained CNNs** (ResNet50, EfficientNet, DenseNet121 pretrained on ImageNet, fine-tuned on IDC) often outperforms a CNN trained from scratch, especially with limited labeled data.
- **Vision Transformers (ViT)** can match or exceed CNNs on large datasets but typically need more training data or heavy pretraining to do so.
- **Ensemble methods** (e.g., averaging predictions from a CNN + a gradient-boosted model on hand-crafted texture features like Haralick/GLCM features) are common in medical imaging pipelines to boost robustness.
For a dataset of this size (277K images) and this image size (50×50), a CNN — ideally with transfer learning — is the well-justified choice.

**4. What is the accuracy after 5 epochs, 10 epochs?**
See the printed output of the "Q4" cell above (Section 3) for your specific run's numbers — it prints train and validation accuracy at exactly epoch 5 and epoch 10.

**5. Is your model overfitting, underfitting, or optimal? Justify.**
Compare the train vs. validation accuracy/loss curves plotted in Section 3:
- If **train accuracy keeps climbing while validation accuracy plateaus or falls** (growing gap between the two curves), the model is **overfitting** — consider more dropout, stronger data augmentation, or fewer filters/layers.
- If **both train and validation accuracy stay low and close together**, the model is **underfitting** — consider training longer, increasing model capacity, or lowering regularization.
- If **train and validation accuracy converge closely at a high value**, and `EarlyStopping` triggers because validation loss stopped improving (rather than because it got worse), the model is close to **optimal** for its capacity and this dataset.
Write your specific conclusion here once you see your actual curves — the architecture already includes batch normalization, dropout (0.25–0.5), and data augmentation specifically to bias the model away from overfitting given the dataset's size.

**6. How can you use it in real life, if given the chance to step further?**
- **Clinical decision support:** deployed as a second-reader tool that pre-screens whole-slide images and flags high-probability IDC regions for a pathologist's priority review, reducing time-to-diagnosis without replacing human judgment.
- **Triage in low-resource settings:** in regions with a shortage of pathologists, a lightweight version of CancerNet (e.g., quantized and deployed on-device) could provide a preliminary screen to prioritize which biopsies need urgent expert review.
- **Whole-slide aggressiveness mapping:** applying the patch-level classifier across an entire whole-mount slide to generate a heatmap of IDC-positive regions, helping surgeons identify tumor margins.
- **Multi-modal extension:** combining this image-based model with structured patient data (age, genetic markers, prior screening history) in a fused model for more personalized risk scoring.

**7. Chance to step further — use your own imagination.**
Ideas to extend this project:
- **Explainability:** add Grad-CAM visualizations to highlight which regions of each patch most influenced the malignant prediction, building pathologist trust in the model's decisions.
- **Multi-class staging:** extend beyond binary benign/malignant to grade tumor aggressiveness (e.g., low/medium/high grade IDC) as a multi-class problem.
- **Federated learning:** train collaboratively across multiple hospitals' data without any single institution's patient images ever leaving its premises, addressing data privacy while still benefiting from a larger effective dataset.
- **Uncertainty quantification:** use Monte Carlo Dropout or a Bayesian CNN so the model can flag "I'm not confident" cases for mandatory human review, rather than always returning a single confident prediction.
